In [1]:
from Util.Problems import Problem, solution

class P031(Problem):
    number = 31
    title = "Coin Sums"
    description = """<p>In the United Kingdom the currency is made up of pound (£) and pence (p). There are eight coins in general circulation:</p><blockquote>1p, 2p, 5p, 10p, 20p, 50p, £1 (100p), and £2 (200p).</blockquote><p>It is possible to make £2 in the following way:</p><blockquote>1×£1 + 1×50p + 2×20p + 1×5p + 1×2p + 3×1p</blockquote><p>How many different ways can £2 be made using any number of coins?</p>"""
    total_pence = 200

In [2]:
p = P031()
p.describe()

## Problem 31: Coin Sums

<p>In the United Kingdom the currency is made up of pound (£) and pence (p). There are eight coins in general circulation:</p><blockquote>1p, 2p, 5p, 10p, 20p, 50p, £1 (100p), and £2 (200p).</blockquote><p>It is possible to make £2 in the following way:</p><blockquote>1×£1 + 1×50p + 2×20p + 1×5p + 1×2p + 3×1p</blockquote><p>How many different ways can £2 be made using any number of coins?</p>

### Solution notes

Starting with a very bad brute force calculating all solutions by looping through 0-200 pennies, then 0-100 twopences etc. Finally checking if the total sum is equal to 200, if so we save the solution.

In [3]:
@solution(P031, first=True, max_tests= 1, make_fast=True, warmup_args=(P031.total_pence, ))
def brute_force(total_pence):
    results = [[0,0,0,0,0,0,0,1]]

    for pennies in range((total_pence // 1) + 1):
        for twopences in range((total_pence // 2) + 1):
            for shillings in range((total_pence // 5) + 1):
                for ten_pence in range((total_pence // 10) + 1):
                    for twenty_pence in range((total_pence // 20) + 1):
                        for fifty_pence in range((total_pence // 50) + 1):
                            for pound in range((total_pence // 100) + 1):
                                total = pennies + twopences * 2 + shillings * 5 + ten_pence * 10 + twenty_pence * 20 + fifty_pence * 50 + pound * 100
                                if total >= total_pence:
                                    if total == total_pence:
                                        results.append([pennies, twopences, shillings, ten_pence, twenty_pence, fifty_pence, pound, 0])
                                    break

    return len(results)

In [4]:
p.test_once("brute_force")

73682 found after a separate test in 831.174800 ms by brute_force (first)


The sum is now being calculated at each coin, so we can break if the sum is already bigger than 200.

In [5]:
@solution(P031, max_tests= 1, make_fast=True, warmup_args=(P031.total_pence, ))
def optimised_brute_force(total_pence):
    results = [[0,0,0,0,0,0,0,1]]

    def calculate_sum(old_sum, new_coins, coins):
        too_big = False
        new_sum = old_sum + new_coins
        if new_sum >= total_pence:
            if new_sum == total_pence:
                results.append(coins)
            too_big = True
        return new_sum, too_big

    for pennies in range((total_pence // 1) + 1):
        for twopences in range((total_pence // 2) + 1):
            twopence_sum, too_big = calculate_sum(pennies, twopences * 2, [pennies, twopences, 0, 0, 0, 0, 0])
            if too_big: break
            for shillings in range((total_pence // 5) + 1):
                shillings_sum, too_big = calculate_sum(twopence_sum, shillings * 5, [pennies, twopences, shillings, 0, 0, 0, 0])
                if too_big: break
                for ten_pence in range((total_pence // 10) + 1):
                    ten_pence_sum, too_big = calculate_sum(shillings_sum, ten_pence * 10, [pennies, twopences, shillings, ten_pence, 0, 0, 0])
                    if too_big: break
                    for twenty_pence in range((total_pence // 20) + 1):
                        twenty_pence_sum, too_big = calculate_sum(ten_pence_sum, twenty_pence * 20, [pennies, twopences, shillings, ten_pence, twenty_pence, 0, 0])
                        if too_big: break
                        for fifty_pence in range((total_pence // 50) + 1):
                            fifty_pence_sum, too_big = calculate_sum(twenty_pence_sum, fifty_pence * 50, [pennies, twopences, shillings, ten_pence, twenty_pence, fifty_pence, 0])
                            if too_big: break
                            for pound in range((total_pence // 100) + 1):
                                total_sum, too_big = calculate_sum(fifty_pence_sum, pound * 100, [pennies, twopences, shillings, ten_pence, twenty_pence, fifty_pence, pound])
                                if too_big: break

    return len(results)

In [6]:
p.test_all()

73682 found after 1 test in 755.954800 ms by brute_force (first)
73682 found after 1 test in 1027.619400 ms by optimised_brute_force


Reversed the order of the coins, if the pound is the outer loop, the earlier break will be possible more often, saving time. Furthermore, we no longer save all solutions, we just count how many there are. Sadly, we cannot replace the list by an int because numba sucks and does not allow global integers. In general global behaviour is ridiculous, as a list is always global, disguising the need to make other variables global manually.

In [7]:
@solution(P031, make_fast=True, warmup_args=(P031.total_pence, ))
def no_solutions_brute_force(total_pence):
    total = []

    def calculate_sum(old_sum, new_coins):
        too_big = False
        new_sum = old_sum + new_coins
        if new_sum >= total_pence:
            if new_sum == total_pence:
                total.append(1)
            too_big = True
        return new_sum, too_big

    for pound in range((total_pence // 100) + 1):
        for fifty_pence in range((total_pence // 50) + 1):
            fifty_pence_sum, too_big = calculate_sum(pound * 100, fifty_pence * 50)
            if too_big: break
            for twenty_pence in range((total_pence // 20) + 1):
                twenty_pence_sum, too_big = calculate_sum(fifty_pence_sum, twenty_pence * 20)
                if too_big: break
                for ten_pence in range((total_pence // 10) + 1):
                    ten_pence_sum, too_big = calculate_sum(twenty_pence_sum, ten_pence * 10)
                    if too_big: break
                    for shillings in range((total_pence // 5) + 1):
                        shillings_sum, too_big = calculate_sum(ten_pence_sum, shillings * 5)
                        if too_big: break
                        for twopence in range((total_pence // 2) + 1):
                            twopence_sum, too_big = calculate_sum(shillings_sum, twopence * 2)
                            if too_big: break
                            for pennies in range(total_pence + 1):
                                total_sum, too_big = calculate_sum(twopence_sum, pennies)
                                if too_big: break

    return len(total)

In [8]:
p.test_once("no_solutions_brute_force")

73681 found after a separate test in 0.240300 ms by no_solutions_brute_force


As numba is dumb, I forced a global variable through using it as an argument and then again as a returned value. Strange but works.

In [9]:
@solution(P031, best= True, make_fast=True, warmup_args=(P031.total_pence, ))
def integer_count_brute_force(total_pence):
    total = 0

    def calculate_sum(old_sum, new_coins, current_total):
        too_big = False
        new_sum = old_sum + new_coins
        if new_sum >= total_pence:
            if new_sum == total_pence:
                current_total += 1
            too_big = True
        return new_sum, too_big, current_total

    for pound in range((total_pence // 100) + 1):
        for fifty_pence in range((total_pence // 50) + 1):
            fifty_pence_sum, too_big, total = calculate_sum(pound * 100, fifty_pence * 50, total)
            if too_big: break
            for twenty_pence in range((total_pence // 20) + 1):
                twenty_pence_sum, too_big, total = calculate_sum(fifty_pence_sum, twenty_pence * 20, total)
                if too_big: break
                for ten_pence in range((total_pence // 10) + 1):
                    ten_pence_sum, too_big, total = calculate_sum(twenty_pence_sum, ten_pence * 10, total)
                    if too_big: break
                    for shillings in range((total_pence // 5) + 1):
                        shillings_sum, too_big, total = calculate_sum(ten_pence_sum, shillings * 5, total)
                        if too_big: break
                        for twopence in range((total_pence // 2) + 1):
                            twopence_sum, too_big, total = calculate_sum(shillings_sum, twopence * 2, total)
                            if too_big: break
                            for pennies in range(total_pence + 1):
                                total_sum, too_big, total = calculate_sum(twopence_sum, pennies, total)
                                if too_big: break

    return total

In [10]:
p.test_all()

73682 found after 1 test in 710.105700 ms by brute_force (first)
73681 found after 1000 tests in 0.055326 ms by integer_count_brute_force
73681 found after 1000 tests in 0.149203 ms by no_solutions_brute_force
73682 found after 1 test in 880.890400 ms by optimised_brute_force
